# 🇻🇳 VietPhuc Remix - AI Try-On Backend Server
### Server sinh ảnh thử đồ cổ phục Việt Nam trên GPU Kaggle Miễn phí (Tesla T4 16GB VRAM)

**Hướng dẫn thiết lập 3 bước:**
1. Ở cột bên phải **Notebook Options**, mục **Accelerator** chọn **GPU T4 x 2** (hoặc GPU T4).
2. Mục **Internet** chọn **Internet On**.
3. Bấm **Run All** (hoặc bấm chạy từng Cell bên dưới).
4. Sau khoảng 1-2 phút, Cell cuối cùng sẽ in ra một đường link dạng:
   `👉 ENDPOINT TRY-ON: https://xxxx.trycloudflare.com/tryon`
5. Copy đường link đó dán vào file `.env` của dự án web:
   `VITE_TRYON_API_URL=https://xxxx.trycloudflare.com/tryon`

In [ ]:
# [Bước 1] Cài đặt các thư viện cần thiết
!pip install -q fastapi uvicorn nest_asyncio diffusers transformers accelerate torch pillow pydantic safetensors
!pip install -q insightface onnxruntime-gpu || echo 'Tiếp tục với chế độ SDXL Lightning...'

In [ ]:
# [Bước 2] Tải công cụ mở cổng HTTPS Public (Cloudflare Tunnel - Không cần đăng ký)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

In [ ]:
# [Bước 3] Tải mô hình SDXL Lightning & Khởi tạo API Server
import os, sys, time, base64, io, re, subprocess
from typing import Optional
from pydantic import BaseModel
from PIL import Image
import torch
import nest_asyncio
import uvicorn
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from diffusers import StableDiffusionXLPipeline, UNet2DConditionModel, EulerDiscreteScheduler
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

app = FastAPI(title='VietPhuc AI Try-On API')

app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'>>> Thiết bị GPU: {device.upper()}')

base_model = 'stabilityai/stable-diffusion-xl-base-1.0'
repo_lightning = 'ByteDance/SDXL-Lightning'
ckpt_file = 'sdxl_lightning_4step_unet.safetensors'

print('>>> Đang tải trọng số SDXL Lightning 4-step...')
unet_config = UNet2DConditionModel.load_config(base_model, subfolder='unet')
unet = UNet2DConditionModel.from_config(unet_config).to(device, torch.float16)
ckpt_path = hf_hub_download(repo_lightning, ckpt_file)
unet.load_state_dict(load_file(ckpt_path, device=device))

pipe = StableDiffusionXLPipeline.from_pretrained(
    base_model,
    unet=unet,
    torch_dtype=torch.float16,
    variant='fp16'
).to(device)
pipe.scheduler = EulerDiscreteScheduler.from_config(pipe.scheduler.config, timestep_spacing='trailing')
pipe.enable_attention_slicing()
print('>>> Mô hình sinh ảnh đã sẵn sàng trên GPU!')

class TryOnPayload(BaseModel):
    face_image: Optional[str] = None
    prompt: str
    outfit: Optional[dict] = None
    negative_prompt: Optional[str] = 'ugly, deformed, disfigured, poor details, bad anatomy, blur, low quality'

@app.post('/tryon')
def tryon_endpoint(data: TryOnPayload):
    t0 = time.time()
    print(f'>>> Nhận request: {data.prompt[:80]}...')
    image = pipe(
        prompt=data.prompt,
        negative_prompt=data.negative_prompt,
        num_inference_steps=4,
        guidance_scale=0,
        width=768,
        height=1024
    ).images[0]
    
    buffered = io.BytesIO()
    image.save(buffered, format='PNG')
    img_str = base64.b64encode(buffered.getvalue()).decode('utf-8')
    duration = round(time.time() - t0, 2)
    print(f'>>> Sinh ảnh xong trong {duration}s!')
    return {'status': 'success', 'image_url': f'data:image/png;base64,{img_str}', 'duration_seconds': duration}

@app.get('/')
def root():
    return {'status': 'online', 'service': 'VietPhuc Try-On Kaggle'}

In [ ]:
# [Bước 4] Mở Cloudflare Tunnel và Khởi động Server
port = 8000
proc = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{port}'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

tunnel_url = None
t0 = time.time()
while time.time() - t0 < 30:
    line = proc.stderr.readline()
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group(0)
        break

print('\n' + '='*65)
if tunnel_url:
    print('🎉 VIETPHUC AI TRY-ON SERVER ĐÃ SẴN SÀNG!')
    print(f'👉 PUBLIC URL: {tunnel_url}')
    print(f'👉 ENDPOINT TRY-ON: {tunnel_url}/tryon')
    print('\nCÁCH DÙNG CHO APP WEB VIETPHUC:')
    print(f'Mở file .env trong dự án và dán dòng sau:')
    print(f'VITE_TRYON_API_URL={tunnel_url}/tryon')
else:
    print('⚠️ Chưa bắt được link trycloudflare, kiểm tra log tunnel.')
print('='*65 + '\n')

# Chạy API Server trên event loop của Jupyter (Hỗ trợ Python 3.13)
config = uvicorn.Config(app, host='0.0.0.0', port=port, log_level='info')
server = uvicorn.Server(config)
await server.serve()